In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import re

import mne
import numpy as np
import seaborn as sns
from scipy.stats import beta
import matplotlib.pyplot as plt
import pandas as pd
import textgrid
from pathlib import Path
from tqdm.auto import tqdm

from src.data import get_electrode_df, add_metadata_features
from src.stimuli import POD_dict
from src.utils import concat_csv_with_indices

In [ ]:
score_percentile_threshold = 90
percentile_filter_metric = "roc_auc"

# window_size = 10
window_size = 20

metric = "roc_auc"

tg_dir = "textgrids"
epochs_path = "outputs/epochs_preprocessed"

In [ ]:
all_epoch_paths = list(Path(epochs_path).glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path))
    epochs[subject_name].metadata = add_metadata_features(epochs[subject_name].metadata)

In [ ]:
plot_times = np.arange(340) / 100 - 0.4

In [ ]:
# Compute 95% CI for random binary guessing ROC AUC performance
assert metric == "roc_auc", "Only ROC AUC is supported for random guessing CI"

test_fold_size = 144
a, b = test_fold_size / 2, test_fold_size / 2
random_guessing_lb, random_guessing_ub = beta.ppf(0.025, a, b), beta.ppf(0.975, a, b)
random_guessing_lb, random_guessing_ub

In [ ]:
scores = concat_csv_with_indices(f"outputs/single_electrode_decoding/{window_size}/*/scores.csv")

In [ ]:
windowed_performance = scores.astype({"smin": int, "smax": int})
# Drop those windows which appear not because they were explicitly sampled but because a larger window got chopped at a boundary.
# This happens iff we don't see the window size appear with smin == 0.
windowed_performance["eff_window_size"] = windowed_performance["smax"] - windowed_performance["smin"]
windowed_performance = windowed_performance[window_size == windowed_performance.eff_window_size]
windowed_performance = windowed_performance \
    .groupby(["target", "subject", "electrode_idx", "phoneme_pair", "smin", "smax"])[metric].mean().sort_values()

# # Only include windows which are above the upper bound of random guessing performance
# windowed_performance = windowed_performance[windowed_performance > random_guessing_ub]

windowed_performance = windowed_performance.reset_index()
windowed_performance

In [ ]:
# g = sns.relplot(data=windowed_performance.reset_index(),
#                 x="smax", y="roc_auc", hue="phoneme_pair", col="target",
#                 kind="line", facet_kws={"sharey": False})
# for ax in g.axes.flat:
#     ax.axvline(0, color="gray", linestyle="--")

## Unified clustering

In [ ]:
target_order = sorted(windowed_performance.target.unique())

In [ ]:
scores_by_site_and_target = windowed_performance \
    .groupby(["subject", "electrode_idx", "phoneme_pair", "target"])[metric].max()
scores_by_site = scores_by_site_and_target \
    .groupby(["subject", "electrode_idx", "phoneme_pair"]).mean()
study_sites = scores_by_site[scores_by_site > np.percentile(scores_by_site, score_percentile_threshold)].index

# for the selected electrodes, compute timecourses of decoding performance
#.loc[study_sites] \
perf_series = windowed_performance \
    .set_index(["subject", "electrode_idx", "phoneme_pair"]).loc[study_sites] \
    .set_index(["target", "smin", "smax"], append=True) \
    .sort_index() \
    .groupby(["subject", "electrode_idx", "phoneme_pair"]) \
    .apply(lambda x: x[metric].values)

perf_series_ids, perf_series = perf_series.index, np.stack(perf_series.values)

global_pca = PCA(n_components=6)
global_pca.fit(perf_series)

In [ ]:
def get_suprathreshold_boundaries(arr, threshold):
    # Boolean mask where condition is met
    mask = arr > threshold

    # Find the change points (rising and falling edges)
    diff = np.diff(mask.astype(int))
    start_indices = np.where(diff == 1)[0] + 1
    end_indices = np.where(diff == -1)[0] + 1

    # Edge case: starts above threshold
    if mask[0]:
        start_indices = np.r_[0, start_indices]
    # Edge case: ends above threshold
    if mask[-1]:
        end_indices = np.r_[end_indices, len(arr)]

    # Combine into (start, end) pairs
    segments = list(zip(start_indices, end_indices))

    return segments

In [ ]:
from matplotlib import transforms
def run_unified_clustering(target_phoneme_pair, plot_tmin=None, plot_tmax=None):
    select_idxs = np.array([i for i, (_, _, phoneme_pair) in enumerate(perf_series_ids)
                            if phoneme_pair == target_phoneme_pair])
    coefs = perf_series[select_idxs]
    coef_ids = [perf_series_ids[i] for i in select_idxs]

    pca = global_pca
    coefs_pca = global_pca.transform(coefs)
    print(f"PCA explained variance: {pca.explained_variance_ratio_}")

    km = KMeans(n_clusters=8, n_init="auto")
    km.fit(coefs_pca)
    print(pd.Series(km.labels_).value_counts().sort_index())

    tg_files = sorted([(int(re.findall(r"_(\d+)\.TextGrid", p.name)[0].lstrip("0")), p) for p in Path(tg_dir).glob("*.TextGrid")
                        if target_phoneme_pair in p.stem])
    # Take first and last stimulus == opposite ends of spectrum
    tg_files = [tg_files[0][1], tg_files[-1][1]]
    tgs = [textgrid.TextGrid.fromFile(tg_file) for tg_file in tg_files]

    ###

    cluster_df = pd.DataFrame([
        {"subject": subject, "electrode_idx": electrode_idx, "phoneme_pair": phoneme_pair,
            "cluster": cluster_idx,
            **{f"pca_{j}": pca_val for j, pca_val in enumerate(coefs_pca[i])}}
        for i, ((subject, electrode_idx, phoneme_pair), cluster_idx) in enumerate(zip(coef_ids, km.labels_))
    ])
    cluster_df = pd.merge(cluster_df, scores_by_site_and_target.unstack("target"),
                            left_on=["subject", "electrode_idx", "phoneme_pair"],
                            right_index=True)
    target_columns = scores_by_site_and_target.index.get_level_values("target").unique()
    cluster_avg_performance = cluster_df.groupby(["cluster"])[target_columns].mean()
    cluster_plot_order = cluster_avg_performance.mean(axis=1).sort_values(ascending=False).index
    cluster_sizes = cluster_df.groupby("cluster").apply(lambda xs: len(xs[["electrode_idx", "phoneme_pair", "subject"]].drop_duplicates()))

    ###

    from matplotlib import transforms

    n_cols = len(target_order)
    n_rows = km.n_clusters

    f, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

    plot_times = (np.array(sorted(windowed_performance.smax.unique())) - 40) / 100

    for axs_i, cluster_idx in zip(axes, cluster_plot_order):
        cluster_idxs = km.labels_ == cluster_idx
        cluster_series = coefs[cluster_idxs]

        cluster_target_series = np.split(cluster_series, len(target_order), axis=1)
        for ax, target, target_series in zip(axs_i, target_order, cluster_target_series):
            mean_series = target_series.mean(axis=0)
            ax.plot(plot_times, mean_series, linewidth=3, color="black")
            ax.set_title(f"{target_phoneme_pair} {target}, Cluster {cluster_idx} ({cluster_sizes.loc[cluster_idx]}, {cluster_avg_performance.loc[cluster_idx, target]:.2f})")

            # indicate above-threshold segments
            segments = get_suprathreshold_boundaries(mean_series, random_guessing_ub)
            for start, end in segments:
                ax.axvspan(plot_times[start], plot_times[end], color="gray", alpha=0.2)
                # label
                ax.text(plot_times[start], 0.8, f"{plot_times[start]} - {plot_times[end]}", rotation=90,
                        transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))

            y_offset = 0
            for tg in tgs:
                intervals = [interval for interval in tg.tiers[0].intervals
                            if interval.mark is not None and interval.mark.strip()]
                for i, interval in enumerate(intervals):
                    if interval.mark is None or not interval.mark.strip():
                            continue
                    ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
                    ax.text(interval.minTime, 0.025 + y_offset, interval.mark.strip(), rotation=90,
                            ha="right", va="bottom",
                            transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))
                    
                    if i == len(intervals) - 1:
                        # plot offset as well.
                        ax.axvline(interval.maxTime, linestyle="--", alpha=0.5, color="blue")
                    
                y_offset += 0.07

            plot_sample = np.random.choice(target_series.shape[0], 10)
            for i in plot_sample:
                ax.plot(plot_times, target_series[i], alpha=0.3)

            ax.axhspan(random_guessing_lb, random_guessing_ub, color="gray", alpha=0.2,)

            if plot_tmin is not None:
                 ax.set_xlim(plot_tmin, ax.get_xlim()[1])
            if plot_tmax is not None:
                 ax.set_xlim(ax.get_xlim()[0], plot_tmax)

    ret = cluster_df
    ret["cluster"] = ret.cluster.astype(str)
    return ret

In [ ]:
all_results = {
    phoneme_pair: run_unified_clustering(phoneme_pair)#, plot_tmax=1.5)
    for phoneme_pair in tqdm(sorted(windowed_performance.phoneme_pair.unique()))
}

In [ ]:
all_results_df = pd.concat(all_results.values())

In [ ]:
causal_decoding_candidates = {
    "bm": {
        "acoustic_primary": [
            (3, 0.1, 0.6),
            (6, 0.15, 0.45),
        ],
        "lex_primary": [
            (1, 0.45, 1.0),
            (7, 0.45, 0.9),
            (4, 0.65, 0.9),
        ],
    },
    "pb": {
        "acoustic_primary": [
            (3, 0.1, 0.45),
        ],
        "lex_primary": [
            (5, 0.45, 0.9),
            (1, 0.45, 1.0),
            (7, 0.35, 0.75),
            (4, 0.9, 1.5),
        ],
    },
    "dn": {
        "acoustic_primary": [
            (5, 0.1, 0.5),
            (0, 0.1, 0.55),
        ],
        "lex_primary": [
            (1, 1.1, 1.65),
            (6, 0.4, 0.9),
            (4, 0.45, 1.2),
            (3, 1.15, 1.8),
            (2, 1.0, 1.4),
        ],
    }
}

causal_decoding_candidates = {
    (phoneme_pair, cluster_a, (a_start, a_end), cluster_b, (b_start, b_end))
    for phoneme_pair, clusters in causal_decoding_candidates.items()
    for cluster_a, a_start, a_end in clusters["acoustic_primary"]
    for cluster_b, b_start, b_end in clusters["lex_primary"]
    if cluster_a != cluster_b
    and a_start < b_start and a_end < b_start
}

expl = all_results_df.set_index(["phoneme_pair", "cluster"]).sort_index()
causal_candidate_populations = []

for phoneme_pair, cluster_a, (a_start, a_end), cluster_b, (b_start, b_end) in causal_decoding_candidates:
    cluster_a_df = expl.loc[(phoneme_pair, str(cluster_a))]
    cluster_b_df = expl.loc[(phoneme_pair, str(cluster_b))]

    matched_subjects = set(cluster_a_df.subject).intersection(cluster_b_df.subject)

    for subject in matched_subjects:
        cluster_a_electrode = cluster_a_df[cluster_a_df.subject == subject]
        cluster_b_electrode = cluster_b_df[cluster_b_df.subject == subject]

        causal_candidate_populations.append({
            "phoneme_pair": phoneme_pair,
            "subject": subject,
            "cluster_a": cluster_a,
            "cluster_b": cluster_b,
            "electrodes_a": cluster_a_df.loc[cluster_a_df.subject == subject].electrode_idx.tolist(),
            "electrodes_b": cluster_b_df.loc[cluster_b_df.subject == subject].electrode_idx.tolist(),
            "window_a": (a_start, a_end),
            "window_b": (b_start, b_end),
        })

In [ ]:
len(causal_candidate_populations)

In [ ]:
import json
with open("causal_candidate_populations.json", "w") as f:
    json.dump(
        sorted(causal_candidate_populations, key=lambda x: -np.mean([len(x["electrodes_a"]), len(x["electrodes_b"])])),
        f)

### Merge with qual

In [ ]:
elec_condition_filters = {
    "stack": [
        ("EC243", 102, "bm"),
        ("EC260", 219, "bm"),
        ("EC243", 103, "bm"),
        ("EC243", 103, "pb"),
        ("EC260", 222, "pb"),
        ("EC243", 105, "bm"),
        ("EC260", 220, "pb"),
        ("EC279", 6, "dn"),
        ("EC248", 365, "dn"),
        ("EC253", 196, "dn"),
        ("EC253", 2, "dn"),
        ("EC279", 167, "dn"),
        ("EC287", 59, "dn"),
        ("EC279", 4, "bm"),
        ("EC279", 4, "dn"),
        ("EC278", 27, "bm"),
        ("EC260", 206, "bm"),
        ("EC260", 206, "pb"),
        ("EC278", 90, "dn"),
    ],

    "alligator": [
        ("EC243", 102, "dn"),
        ("EC243", 102, "pb"),
        ("EC260", 204, "dn"),
        ("EC260", 91, "dn"),
        ("EC260", 93, "dn"),
        ("EC243", 197, "bm"),
        ("EC260", 109, "dn"),
        ("EC243", 103, "dn"),
        ("EC278", 121, "bm"),
        ("EC260", 92, "dn"),
        ("EC250", 216, "pb"),
        ("EC243", 213, "dn"),
        ("EC248", 364, "dn"),
        ("EC250", 207, "dn"),
        ("EC248", 253, "dn"),
        ("EC282", 97, "dn"),
        ("EC278", 27, "dn"),
        ("EC282", 115, "pb"),
        ("EC278", 90, "pb"),
        ("EC279", 76, "bm"),
    ],

    "loo": [
        ('EC260', 204, 'dn'),
        ('EC260', 204, 'pb'),
        ('EC260', 91, 'bm'),
        ('EC260', 93, 'pb'),
        ('EC243', 197, 'dn'),
        ('EC260', 219, 'bm'),
        ('EC260', 219, 'dn'),
        ('EC278', 121, 'dn'),
        ('EC243', 119, 'bm'),
        ('EC243', 119, 'dn'),
        ('EC260', 222, 'bm'),
        ('EC260', 222, 'dn'),
        ('EC243', 105, 'pb'),
        ('EC260', 92, 'bm'),
        ('EC250', 216, 'bm'),
        ('EC260', 221, 'dn'),
        ('EC260', 221, 'pb'),
        ('EC248', 381, 'dn'),
        ('EC260', 220, 'dn'),
        ('EC253', 212, 'dn'),
        ('EC250', 215, 'bm'),
        ('EC250', 215, 'dn'),
        ('EC248', 364, 'bm'),
        ('EC260', 76, 'bm'),
        ('EC260', 76, 'dn'),
        ('EC270', 122, 'dn'),
        ('EC282', 116, 'bm'),
        ('EC287', 124, 'pb'),
        ('EC253', 196, 'pb'),
        ('EC248', 253, 'dn'),
        ('EC279', 167, 'pb'),
        ('EC279', 152, 'dn'),
        ('EC279', 152, 'pb'),
        ('EC287', 5, 'pb'),
        ('EC270', 140, 'dn'),
        ('EC260', 206, 'dn'),
        ('EC278', 90, 'bm'),
        ('EC243', 228, 'dn'),
        ('EC243', 72, 'bm'),
        ('EC243', 72, 'dn'),
        ('EC279', 11, 'pb'),
        ('EC279', 76, 'dn'),
    ]
}

qual_results = pd.concat({
    label: pd.DataFrame(elecs, columns=["subject", "electrode_idx", "phoneme_pair"])
    for label, elecs in elec_condition_filters.items()
}, names=["qual_morph"]).droplevel(-1).reset_index()

qual_results = pd.merge(
    qual_results, scores_by_site_and_target.unstack("target"),
    on=["subject", "electrode_idx", "phoneme_pair"])

dev = pd.merge(all_results_df, qual_results, on=["subject", "electrode_idx", "phoneme_pair"], how="outer")
for target in target_order:
    dev[target] = np.where(dev[f"{target}_x"].isna(), dev[f"{target}_y"], dev[f"{target}_x"])
    dev.drop(columns=[f"{target}_y", f"{target}_x"], inplace=True)
dev["qual_morph"] = dev.qual_morph.fillna("na")

In [ ]:
hue_order = sorted(dev.cluster.value_counts().index)
g = sns.catplot(data=dev.melt(id_vars=sorted(set(dev.columns) - set(target_order)), var_name="target"),
                x="phoneme_pair", hue="cluster", hue_order=hue_order, y="value", col="target", kind="box")
# g.ax.set_ylim(0.48, 1)
# g.ax.axhline(0.5, linestyle="--", color="black")
# g.ax.set_ylabel("ROC/AUC")

In [ ]:
g = sns.catplot(data=dev.melt(id_vars=sorted(set(dev.columns) - set(target_order)), var_name="target"),
                x="phoneme_pair", hue="qual_morph", y="value", col="target", kind="box")
# g.ax.set_ylim(0.48, 1)
# g.ax.axhline(0.5, linestyle="--", color="black")
# g.ax.set_ylabel("ROC/AUC")

## Prepare performance timecourses

In [ ]:
from matplotlib import transforms


def cluster_and_plot(coefs, coef_ids, target_phoneme_pair,
                     plot_times,
                     global_pca, scores_by_site,
                     n_clusters=4,
                     use_global_pca=True,
                     plot_heatmap=True,
                     plot_lineplot=True,):
    assert len(plot_times) == coefs.shape[1]

    select_idxs = np.array([i for i, (_, _, phoneme_pair) in enumerate(coef_ids)
                            if phoneme_pair == target_phoneme_pair])
    coefs = coefs[select_idxs]
    coef_ids = [coef_ids[i] for i in select_idxs]

    if use_global_pca:
        pca = global_pca
        coefs_pca = global_pca.transform(coefs)
    else:
        pca = PCA(n_components=8)
        coefs_pca = pca.fit_transform(coefs)
    print(f"PCA explained variance: {pca.explained_variance_ratio_}")

    km = KMeans(n_clusters=n_clusters, n_init=5)
    km.fit(coefs_pca)
    print(pd.Series(km.labels_).value_counts().sort_index())

    tg_files = sorted([(int(re.findall(r"_(\d+)\.TextGrid", p.name)[0].lstrip("0")), p) for p in Path(tg_dir).glob("*.TextGrid")
                       if target_phoneme_pair in p.stem])
    # Take first and last stimulus == opposite ends of spectrum
    tg_files = [tg_files[0][1], tg_files[-1][1]]
    tgs = [textgrid.TextGrid.fromFile(tg_file) for tg_file in tg_files]
    
    ####

    # # plt.scatter(all_coefs_pca[:, 0], all_coefs_pca[:, 1], c=km.labels_)
    # f, ax = plt.subplots(figsize=(6, 6))
    # for cluster_idx in range(km.n_clusters):
    #     plt.scatter(coefs_pca[km.labels_ == cluster_idx, 0], coefs_pca[km.labels_ == cluster_idx, 1],
    #                 label=f"Cluster {cluster_idx}")
    # plt.legend()

    ####

    # f, ax = plt.subplots(figsize=(8, 4))
    # for pc in range(4):
    #     ax.plot(window_plot_times, pca.components_[pc], label=f"PC {pc}")
    # # plot POD line
    # ax.axvline(POD_dict[target_phoneme_pair], color="gray", linestyle="--")
    # ax.legend(loc="upper right", bbox_to_anchor=(1.2, 1))

    # y_offset = 0
    # for tg in tgs:
    #     intervals = [interval for interval in tg.tiers[0].intervals
    #                  if interval.mark is not None and interval.mark.strip()]
    #     for i, interval in enumerate(intervals):
    #         if interval.mark is None or not interval.mark.strip():
    #                 continue
    #         ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
    #         ax.text(interval.minTime, 0.025 + y_offset, interval.mark.strip(), rotation=90,
    #                 ha="right", va="bottom",
    #                 transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))
            
    #         if i == len(intervals) - 1:
    #             # plot offset as well.
    #             ax.axvline(interval.maxTime, linestyle="--", alpha=0.5, color="blue")
            
    #     y_offset += 0.07

    ####

    cluster_df = pd.DataFrame([
        {"subject": subject, "electrode_idx": electrode_idx, "phoneme_pair": phoneme_pair,
         "cluster": cluster_idx,
         **{f"pca_{j}": pca_val for j, pca_val in enumerate(coefs_pca[i])}}
        for i, ((subject, electrode_idx, phoneme_pair), cluster_idx) in enumerate(zip(coef_ids, km.labels_))
    ])
    cluster_df = pd.merge(cluster_df, scores_by_site,
                          left_on=["subject", "electrode_idx", "phoneme_pair"],
                          right_index=True)
    cluster_avg_performance = cluster_df.groupby("cluster")[metric].mean()
    cluster_plot_order = cluster_avg_performance.sort_values(ascending=False).index
    cluster_sizes = cluster_df.groupby("cluster").size()

    ####

    if plot_heatmap:
        n_cols = 2
        n_rows = (n_clusters + n_cols - 1) // n_cols

        f, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

        for i, (ax, cluster_idx) in enumerate(zip(axes.ravel(), cluster_plot_order)):
            cluster_idxs = km.labels_ == cluster_idx
            cluster_series = coefs[cluster_idxs]

            # resort based on position of peak
            cluster_series = cluster_series[np.argsort(cluster_series.argmax(axis=1))]

            cluster_series = pd.DataFrame(cluster_series)
            cluster_series.columns = np.round(plot_times, 2)
            sns.heatmap(cluster_series, ax=ax, cmap="coolwarm",
                        center=0.5, cbar=i % n_cols == n_cols - 1)
            ax.set_title(f"Cluster {cluster_idx} ({cluster_sizes.loc[cluster_idx]}, {cluster_avg_performance.loc[cluster_idx]:.2f})")
            # remove yticks
            ax.set_yticks([])

    ####

    if plot_lineplot:
        n_cols = 2
        n_rows = (n_clusters + n_cols - 1) // n_cols

        f, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

        for ax, cluster_idx in zip(axes.ravel(), cluster_plot_order):
            cluster_idxs = km.labels_ == cluster_idx
            cluster_series = coefs[cluster_idxs]

            ax.plot(plot_times, cluster_series.mean(axis=0), linewidth=3, color="black")
            ax.set_title(f"{target_phoneme_pair}, Cluster {cluster_idx} ({cluster_sizes.loc[cluster_idx]}, {cluster_avg_performance.loc[cluster_idx]:.2f})")

            y_offset = 0
            for tg in tgs:
                intervals = [interval for interval in tg.tiers[0].intervals
                            if interval.mark is not None and interval.mark.strip()]
                for i, interval in enumerate(intervals):
                    if interval.mark is None or not interval.mark.strip():
                            continue
                    ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
                    ax.text(interval.minTime, 0.025 + y_offset, interval.mark.strip(), rotation=90,
                            ha="right", va="bottom",
                            transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))
                    
                    if i == len(intervals) - 1:
                        # plot offset as well.
                        ax.axvline(interval.maxTime, linestyle="--", alpha=0.5, color="blue")
                    
                y_offset += 0.07

            plot_sample = np.random.choice(cluster_series.shape[0], 10)
            for i in plot_sample:
                ax.plot(plot_times, cluster_series[i], alpha=0.3)

            ax.axhspan(random_guessing_lb, random_guessing_ub, color="gray", alpha=0.2,)

    ret = cluster_df
    ret["cluster"] = ret.cluster.astype(str)
    return ret    

In [ ]:
def analyze_target(target, n_clusters=4, **kwargs):
    # # For each target, sample the top percentile of electrodes on the given metric
    scores_by_site = windowed_performance.query("target == @target") \
        .groupby(["subject", "electrode_idx", "phoneme_pair"])[metric].max()
    study_sites = scores_by_site[scores_by_site > np.percentile(scores_by_site, score_percentile_threshold)].index

    scores_avg_lex = windowed_performance.query("target == @target")

    # for the selected electrodes, compute timecourses of decoding performance
    #.loc[study_sites] \
    perf_series = scores_avg_lex \
        .set_index(["subject", "electrode_idx", "phoneme_pair"]).loc[study_sites] \
        .set_index(["smin", "smax"], append=True) \
        .sort_index() \
        .groupby(["subject", "electrode_idx", "phoneme_pair"]) \
        .apply(lambda x: x[metric].values)

    perf_series_ids, perf_series = perf_series.index, np.stack(perf_series.values)

    global_pca = PCA(n_components=6)
    global_pca.fit(perf_series)

    # HACK manual
    window_plot_times = (np.array(sorted(scores_avg_lex.smax.unique())) - 20) / 120
    assert len(window_plot_times) == perf_series.shape[1]

    results = {}
    for phoneme_pair in study_sites.get_level_values("phoneme_pair").unique():
        print("Phoneme pair:", phoneme_pair)
        results[phoneme_pair] = cluster_and_plot(perf_series, perf_series_ids, phoneme_pair,
                                                 window_plot_times,
                                                 global_pca, scores_by_site,
                                                 n_clusters=n_clusters,
                                                 use_global_pca=True, **kwargs)
        from IPython.core.display import display, HTML
        display(HTML(f"<h1>{phoneme_pair}</h1>"))
        
    # combine results
    results = pd.concat(results.values())
    results = results.reset_index(drop=True)
    results["target"] = target
    results["metric"] = metric

    return results

In [ ]:
def plot_target_response(target, target_phoneme_pair, sites):
    # for the selected electrodes, compute timecourses of decoding performance
    perf_series = windowed_performance.query("target == @target") \
        .set_index(["subject", "electrode_idx", "phoneme_pair"]).loc[sites] \
        .set_index(["smin", "smax"], append=True) \
        .sort_index() \
        .groupby(["subject", "electrode_idx", "phoneme_pair"]) \
        .apply(lambda x: x[metric].values)
    
    perf_series_ids, perf_series = perf_series.index, np.stack(perf_series.values)

    window_plot_times = sorted(windowed_performance.t_center.unique())
    assert len(window_plot_times) == perf_series.shape[1]

    ####

    tg_files = sorted([(int(re.findall(r"_(\d+)\.TextGrid", p.name)[0].lstrip("0")), p)
                       for p in Path(tg_dir).glob("*.TextGrid")
                       if target_phoneme_pair in p.stem])
    # Take first and last stimulus == opposite ends of spectrum
    tg_files = [tg_files[0][1], tg_files[-1][1]]
    tgs = [textgrid.TextGrid.fromFile(tg_file) for tg_file in tg_files]

    ####

    f, ax = plt.subplots(figsize=(7, 3))

    ys = perf_series

    ax.plot(window_plot_times, ys.mean(axis=0), linewidth=3, color="black")
    ax.set_title(f"{len(ys)} electrodes, {target_phoneme_pair}")

    y_offset = 0
    for tg in tgs:
        intervals = [interval for interval in tg.tiers[0].intervals
                    if interval.mark is not None and interval.mark.strip()]
        for i, interval in enumerate(intervals):
            if interval.mark is None or not interval.mark.strip():
                    continue
            ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
            ax.text(interval.minTime, 0.025 + y_offset, interval.mark.strip(), rotation=90,
                    ha="right", va="bottom",
                    transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))
            
            if i == len(intervals) - 1:
                # plot offset as well.
                ax.axvline(interval.maxTime, linestyle="--", alpha=0.5, color="blue")
            
        y_offset += 0.07

    plot_sample = ys
    for i in range(len(plot_sample)):
        ax.plot(window_plot_times, ys[i], alpha=0.3)

In [ ]:
all_cluster_results_arr = []

In [ ]:
all_cluster_results_arr.append(analyze_target("lexical_evidence", n_clusters=6, plot_heatmap=False))

In [ ]:
all_cluster_results_arr.append(analyze_target("mismatch", n_clusters=6, plot_heatmap=False))

In [ ]:
all_cluster_results_arr.append(analyze_target("mismatch_left_right", n_clusters=8, plot_heatmap=False))

In [ ]:
all_cluster_results = pd.concat(all_cluster_results_arr)

In [ ]:
all_cluster_results.query("subject == 'EC243' and electrode_idx == 103")

## Plot trial rasters for electrodes of interest

In [ ]:
all_cluster_results

In [ ]:
def plot_eoi_rasters(eois, sort_order=("phoneme_pair", "word_end", "resampled"), clip=6):
    n_cols = 2
    n_rows = int(np.ceil(len(eois) / n_cols))
    f, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

    for ax, eoi in zip(axes.ravel(), eois):
        # get the electrode
        eoi = tuple(eoi)
        md = epochs[eoi[0]].metadata.rename_axis("epoch_idx").reset_index().set_index(list(sort_order)).sort_index()
        data = epochs[eoi[0]].copy()[md.epoch_idx.values].pick(eoi[1]).get_data()[:, 0, :]
        if clip is not None:
            data = np.clip(data, -clip, clip)
        sns.heatmap(data, ax=ax)
        ax.set_title(f"{eoi[0]}, {eoi[1]}")
        ax.set_yticks([])
        ax.set_xticklabels(["%.2f" % plot_times[int(t)] for t in ax.get_xticks()], rotation=45, ha="right")
        ax.set_xlabel("Time (s)")

        sep_index = md.index.droplevel("resampled")
        sep_pos = np.where(sep_index != sep_index.to_series().shift(1))[0]
        for sep in sep_pos:
            ax.axhline(sep, color="red", linestyle="--")
            ax.text(-0.1, sep + 0.5, md.iloc[sep].name[1], ha="right", va="center",
                    transform=transforms.blended_transform_factory(ax.transAxes, ax.transData))
            
    f.tight_layout()
    # md = epochs[eoi[0]].metadata.rename_axis("epoch_idx").reset_index().set_index(["phoneme_pair", "resampled", "word_end"]).sort_index()
    # sns.heatmap(epochs[eoi[0]].copy()[md.epoch_idx.values].pick(eoi[1]).get_data()[:, 0, :])

In [ ]:
eois = all_cluster_results.query("phoneme_pair == 'dn' and target == 'lexical_evidence' and cluster == '0'").sort_values("roc_auc").iloc[-6:][["subject", "electrode_idx"]].values

plot_eoi_rasters(eois)

## Prepare coefficients

This section uses the old method of comparing beta weights across encoders using the entire time series to predict binary labels.

In [ ]:
# coef_ids = [
#     (subject, electrode_idx, phoneme_pair, smin, smax, fold_idx)
#     for subject, electrode_idx, phoneme_pair, smin, smax in results["models"].keys()
#     for fold_idx in range(len(results["models"][subject, electrode_idx, phoneme_pair, smin, smax]))
#     if (subject, electrode_idx, phoneme_pair, smin, smax) in study_scores.index
# ]

In [ ]:
# coef_ids = [
#     (subject, electrode_idx, phoneme_pair)
#     for subject, electrode_idx, phoneme_pair in results["models"].keys()
#     if (subject, electrode_idx, phoneme_pair) in study_scores.index
# ]

# num_folds = len(next(iter(results["models"])))

# all_coefs = np.array([
#     [results["models"][subject, electrode_idx, phoneme_pair][fold_idx].steps[-1][1].coef_.squeeze()
#      for fold_idx in folds]
#     for subject, electrode_idx, phoneme_pair in coef_ids
# ])

# # take mean over electrodes
# all_coefs = all_coefs.mean(axis=1)

In [ ]:
# all_coefs_norm = all_coefs / np.abs(all_coefs).max(1, keepdims=True)

In [ ]:
# global_pca = PCA(n_components=8)
# global_pca.fit(all_coefs_norm)

In [ ]:
# from matplotlib import transforms


# def cluster_and_plot(coefs, coef_ids, target_phoneme_pair, n_clusters=4,
#                      use_global_pca=True):
#     select_idxs = np.array([i for i, (_, _, phoneme_pair) in enumerate(coef_ids)
#                             if phoneme_pair == target_phoneme_pair])
#     coefs = coefs[select_idxs]
#     coef_ids = [coef_ids[i] for i in select_idxs]

#     if use_global_pca:
#         pca = global_pca
#         coefs_pca = global_pca.transform(coefs)
#     else:
#         pca = PCA(n_components=8)
#         coefs_pca = pca.fit_transform(coefs)
#     print(f"PCA explained variance: {pca.explained_variance_ratio_}")

#     km = KMeans(n_clusters=n_clusters, n_init=5)
#     km.fit(coefs_pca)
#     print(pd.Series(km.labels_).value_counts().sort_index())


#     tg_file = next(p for p in Path(tg_dir).glob("*.TextGrid") if target_phoneme_pair in p.stem)
#     tg = textgrid.TextGrid.fromFile(tg_file)
    
#     ####

#     # plt.scatter(all_coefs_pca[:, 0], all_coefs_pca[:, 1], c=km.labels_)
#     f, ax = plt.subplots(figsize=(6, 6))
#     for cluster_idx in range(km.n_clusters):
#         plt.scatter(coefs_pca[km.labels_ == cluster_idx, 0], coefs_pca[km.labels_ == cluster_idx, 1],
#                     label=f"Cluster {cluster_idx}")
#     plt.legend()

#     ####

#     f, ax = plt.subplots(figsize=(8, 4))
#     for pc in range(4):
#         ax.plot(plot_times, pca.components_[pc], label=f"PC {pc}")
#     ax.legend(loc="upper right", bbox_to_anchor=(1.2, 1))

#     for interval in tg.tiers[0].intervals:
#         if interval.mark is None or not interval.mark.strip():
#                 continue
#         ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
#         ax.text(interval.minTime, 0.025, interval.mark.strip(), rotation=90,
#                 ha="right", va="bottom",
#                 transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))

#     ####

#     cluster_df = pd.DataFrame([
#         {"subject": subject, "electrode_idx": electrode_idx, "phoneme_pair": phoneme_pair,
#          "cluster": cluster_idx,
#          **{f"pca_{j}": pca_val for j, pca_val in enumerate(coefs_pca[i])}}
#         for i, ((subject, electrode_idx, phoneme_pair), cluster_idx) in enumerate(zip(coef_ids, km.labels_))
#     ])
#     cluster_df = pd.merge(cluster_df, scores_avg,
#                           left_on=["subject", "electrode_idx", "phoneme_pair"],
#                           right_index=True)
#     cluster_avg_performance = cluster_df.groupby("cluster").roc_auc.mean()
#     cluster_plot_order = cluster_avg_performance.sort_values(ascending=False).index
#     cluster_sizes = cluster_df.groupby("cluster").size()

#     ####

#     n_cols = 2
#     n_rows = (n_clusters + n_cols - 1) // n_cols

#     f, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

#     for i, (ax, cluster_idx) in enumerate(zip(axes.ravel(), cluster_plot_order)):
#         cluster_idxs = km.labels_ == cluster_idx
#         cluster_series = coefs[cluster_idxs]

#         # resort based on position of peak
#         cluster_series = cluster_series[np.argsort(cluster_series.argmax(axis=1))]

#         sns.heatmap(cluster_series, ax=ax, cmap="coolwarm", center=0, cbar=i % n_cols == n_cols - 1)
#         ax.set_title(f"Cluster {cluster_idx} ({cluster_sizes.loc[cluster_idx]}, {cluster_avg_performance.loc[cluster_idx]:.2f})")
#         # remove yticks
#         ax.set_yticks([])

#     ####

#     n_cols = 2
#     n_rows = (n_clusters + n_cols - 1) // n_cols

#     f, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

#     for ax, cluster_idx in zip(axes.ravel(), cluster_plot_order):
#         cluster_idxs = km.labels_ == cluster_idx
#         cluster_series = coefs[cluster_idxs]

#         ax.plot(plot_times, cluster_series.mean(axis=0), linewidth=3, color="black")
#         ax.set_title(f"Cluster {cluster_idx} ({cluster_sizes.loc[cluster_idx]}, {cluster_avg_performance.loc[cluster_idx]:.2f})")

#         for interval in tg.tiers[0].intervals:
#             if interval.mark is None or not interval.mark.strip():
#                     continue
#             ax.axvline(interval.minTime, linestyle="--", alpha=0.5, color="salmon")
#             ax.text(interval.minTime, 0.025, interval.mark.strip(), rotation=90,
#                     ha="right", va="bottom",
#                     transform=transforms.blended_transform_factory(ax.transData, ax.transAxes))

#         plot_sample = np.random.choice(cluster_series.shape[0], 10)
#         for i in plot_sample:
#             ax.plot(plot_times, cluster_series[i], alpha=0.3)

#     ret = cluster_df
#     ret["cluster"] = ret.cluster.astype(str)
#     return ret    

In [ ]:
# results_bm = cluster_and_plot(all_coefs_norm, coef_ids, "bm")

In [ ]:
# results_pb = cluster_and_plot(all_coefs_norm, coef_ids, "pb")

In [ ]:
# results_dn = cluster_and_plot(all_coefs_norm, coef_ids, "dn")

## Merge

In [ ]:
all_cluster_results = pd.concat(all_cluster_results_arr)

In [ ]:
all_cluster_results.groupby(["phoneme_pair", "cluster"])[metric].mean().sort_values()

In [ ]:
all_cluster_results

In [ ]:
explore = all_cluster_results[~all_cluster_results.cluster.isna()].astype({"cluster": int}).pivot_table(index=["subject", "electrode_idx", "phoneme_pair"], columns=["target"], values=["cluster"])
explore.columns = explore.columns.droplevel(0)

In [ ]:
epochs["EC243"].metadata.query("phoneme_pair == 'bm'")

In [ ]:
shared_mismatch = (~explore.mismatch.unstack("phoneme_pair").isna()).sum(1)
shared_mismatch = shared_mismatch[shared_mismatch >= 2].sort_index()
shared_mismatch

In [ ]:
explore.xs("dn", level="phoneme_pair").query("lexical_evidence == 5")

In [ ]:
plot_target_response("mismatch_left_right", "pb", explore.xs("pb", level="phoneme_pair", drop_level=False).query("mismatch_left_right == 3 and not lexical_evidence.isnull()").index)

In [ ]:
plot_target_response("mismatch_left_right", "pb", explore.xs("pb", level="phoneme_pair", drop_level=False).query("mismatch_left_right == 3 and lexical_evidence.isnull()").index)

In [ ]:
explore.xs("dn", level="phoneme_pair").query("mismatch_left_right == 1")

In [ ]:
plot_target_response("mismatch_left_right", "dn", explore.xs("dn", level="phoneme_pair", drop_level=False).query("mismatch_left_right == 1 and lexical_evidence.isnull()").index)

In [ ]:
plot_target_response("mismatch_left_right", "dn", explore.xs("dn", level="phoneme_pair", drop_level=False).query("mismatch_left_right == 1 and not lexical_evidence.isnull()").index)

In [ ]:
plot_target_response("mismatch_left_right", "dn", explore.xs("dn", level="phoneme_pair", drop_level=False).query("mismatch_left_right == 3 and lexical_evidence.isnull()").index)

In [ ]:
plot_target_response("mismatch_left_right", "dn", explore.xs("dn", level="phoneme_pair", drop_level=False).query("mismatch_left_right == 3 and not lexical_evidence.isnull()").index)

In [ ]:
explore

In [ ]:
# Convert to boolean masks for null values
lexical_null = explore["lexical_evidence"].isnull()
mismatch_null = explore["mismatch"].isnull()
mismatch_lr_null = explore["mismatch_left_right"].isnull()

# Count occurrences for Venn diagram
set_sizes = {
    "Lexical": lexical_null.sum(),
    "Mismatch": mismatch_null.sum(),
    "Mismatch Left-Right": mismatch_lr_null.sum(),
    "Lexical & Mismatch": (lexical_null & mismatch_null).sum(),
    "Lexical & Mismatch Left-Right": (lexical_null & mismatch_lr_null).sum(),
    "Mismatch & Mismatch Left-Right": (mismatch_null & mismatch_lr_null).sum(),
    "All Three": (lexical_null & mismatch_null & mismatch_lr_null).sum(),
}

# Create the Venn diagram
plt.figure(figsize=(6,6))
from matplotlib_venn import venn3
venn = venn3(subsets=(
    set_sizes["Lexical"], 
    set_sizes["Mismatch"], 
    set_sizes["Lexical & Mismatch"], 
    set_sizes["Mismatch Left-Right"], 
    set_sizes["Lexical & Mismatch Left-Right"], 
    set_sizes["Mismatch & Mismatch Left-Right"], 
    set_sizes["All Three"]
), set_labels=("Lexical", "Mismatch", "Mismatch Left-Right"))

### Compare with qualitative labeling

In [ ]:
elec_condition_filters = {
    "stack": [
        ("EC243", 102, "bm"),
        ("EC260", 219, "bm"),
        ("EC243", 103, "bm"),
        ("EC243", 103, "pb"),
        ("EC260", 222, "pb"),
        ("EC243", 105, "bm"),
        ("EC260", 220, "pb"),
        ("EC279", 6, "dn"),
        ("EC248", 365, "dn"),
        ("EC253", 196, "dn"),
        ("EC253", 2, "dn"),
        ("EC279", 167, "dn"),
        ("EC287", 59, "dn"),
        ("EC279", 4, "bm"),
        ("EC279", 4, "dn"),
        ("EC278", 27, "bm"),
        ("EC260", 206, "bm"),
        ("EC260", 206, "pb"),
        ("EC278", 90, "dn"),
    ],

    "alligator": [
        ("EC243", 102, "dn"),
        ("EC243", 102, "pb"),
        ("EC260", 204, "dn"),
        ("EC260", 91, "dn"),
        ("EC260", 93, "dn"),
        ("EC243", 197, "bm"),
        ("EC260", 109, "dn"),
        ("EC243", 103, "dn"),
        ("EC278", 121, "bm"),
        ("EC260", 92, "dn"),
        ("EC250", 216, "pb"),
        ("EC243", 213, "dn"),
        ("EC248", 364, "dn"),
        ("EC250", 207, "dn"),
        ("EC248", 253, "dn"),
        ("EC282", 97, "dn"),
        ("EC278", 27, "dn"),
        ("EC282", 115, "pb"),
        ("EC278", 90, "pb"),
        ("EC279", 76, "bm"),
    ],

    "loo": [
        ('EC260', 204, 'dn'),
        ('EC260', 204, 'pb'),
        ('EC260', 91, 'bm'),
        ('EC260', 93, 'pb'),
        ('EC243', 197, 'dn'),
        ('EC260', 219, 'bm'),
        ('EC260', 219, 'dn'),
        ('EC278', 121, 'dn'),
        ('EC243', 119, 'bm'),
        ('EC243', 119, 'dn'),
        ('EC260', 222, 'bm'),
        ('EC260', 222, 'dn'),
        ('EC243', 105, 'pb'),
        ('EC260', 92, 'bm'),
        ('EC250', 216, 'bm'),
        ('EC260', 221, 'dn'),
        ('EC260', 221, 'pb'),
        ('EC248', 381, 'dn'),
        ('EC260', 220, 'dn'),
        ('EC253', 212, 'dn'),
        ('EC250', 215, 'bm'),
        ('EC250', 215, 'dn'),
        ('EC248', 364, 'bm'),
        ('EC260', 76, 'bm'),
        ('EC260', 76, 'dn'),
        ('EC270', 122, 'dn'),
        ('EC282', 116, 'bm'),
        ('EC287', 124, 'pb'),
        ('EC253', 196, 'pb'),
        ('EC248', 253, 'dn'),
        ('EC279', 167, 'pb'),
        ('EC279', 152, 'dn'),
        ('EC279', 152, 'pb'),
        ('EC287', 5, 'pb'),
        ('EC270', 140, 'dn'),
        ('EC260', 206, 'dn'),
        ('EC278', 90, 'bm'),
        ('EC243', 228, 'dn'),
        ('EC243', 72, 'bm'),
        ('EC243', 72, 'dn'),
        ('EC279', 11, 'pb'),
        ('EC279', 76, 'dn'),
    ]
}

qual_results = pd.concat({
    label: pd.DataFrame(elecs, columns=["subject", "electrode_idx", "phoneme_pair"])
    for label, elecs in elec_condition_filters.items()
}, names=["qual_morph"]).droplevel(-1).reset_index()

qual_results = pd.merge(
    qual_results, scores_avg.query("variable == 'roc_auc'").groupby(["target", "subject", "electrode_idx", "phoneme_pair"]).value.max().reset_index(),
    on=["subject", "electrode_idx", "phoneme_pair"])

all_cluster_results = pd.merge(all_cluster_results, qual_results, on=["target", "subject", "electrode_idx", "phoneme_pair"], how="outer")
all_cluster_results["value"] = np.where(all_cluster_results.value_x.isna(), all_cluster_results.value_y, all_cluster_results.value_x)
all_cluster_results["qual_morph"] = all_cluster_results.qual_morph.fillna("na")

In [ ]:
hue_order = sorted(all_cluster_results.cluster.value_counts().index)
g = sns.catplot(data=all_cluster_results, x="phoneme_pair", hue="cluster", hue_order=hue_order, y="value", row="target", kind="box")
# g.ax.set_ylim(0.48, 1)
# g.ax.axhline(0.5, linestyle="--", color="black")
# g.ax.set_ylabel("ROC/AUC")

In [ ]:
sns.catplot(data=all_cluster_results.fillna({"cluster": "na"}).groupby(["target", "phoneme_pair", "cluster"]).qual_morph.value_counts().reset_index(),
            x="cluster", hue="qual_morph", y="count", row="phoneme_pair", col="target", kind="bar", height=2, aspect=2,
            sharey=False)

In [ ]:
g = sns.catplot(data=all_cluster_results, x="phoneme_pair", hue="qual_morph", y="value", row="target", kind="box")
# g.ax.set_ylim(0.48, 1)
# g.ax.axhline(0.5, linestyle="--", color="black")
# g.ax.set_ylabel("ROC/AUC")

In [ ]:
all_cluster_results.fillna({"cluster": "na"}).groupby(["target", "phoneme_pair", "qual_morph", "cluster"]).size().unstack("cluster").fillna(0).astype(int)

In [ ]:
all_cluster_results.to_csv("single_electrode_clustering.csv")